# 再現可能なデータ分割

**学習目標**
 * train/valid/test 分割のさまざまな作成方法がもたらす影響を探索する

## 概要
    
機械学習において再現性（Repeatability）は非常に重要です。今実行したことと 5 分後に実行したことで異なる結果が得られるようでは、実験を進めることが困難になります。言い換えると、自分が行った変更によってモデルが改善されたのかどうかを判断するのが難しくなります。

In [ ]:
from google.cloud import bigquery

<h3> シンプルな機械学習モデルの作成 </h3>

使用するデータセットは、航空会社の到着データに関する <a href="https://bigquery.cloud.google.com/table/bigquery-samples:airline_ontime_data.flights">BigQuery の一般公開データセット</a> です。リンクをクリックして、列名を確認してください。[詳細] タブに切り替えてレコード数が 7,000 万件であることを確認し、[プレビュー] タブに切り替えていくつかの行を確認します。
<p>
出発遅延（departure delay）に基づいて航空便の到着遅延（arrival delay）を予測したいとします。使用するモデルは、バイアス項がゼロの線形モデルです。
$$ delay_{arrival} = \alpha * delay_{departure} $$
<p>
モデルをトレーニングするということは、$\alpha$ の適切な値を推定することです。
<p>
$\alpha$ を推定するアプローチの 1 つは、次の式を使用することです。
$$ \alpha = \frac{\sum delay_{departure} delay_{arrival} }{  \sum delay_{departure}^2 } $$
ニューヨーク発ロサンゼルス行きと、オースティン発インディアナポリス行き（飛行時間が短く、混雑していない空港）ではこの関係が異なるという考え方を反映させたいため、空港ペアごとに異なる $\alpha$ を計算します。簡単のため、ここではデンバーとロサンゼルス間のフライトのみを対象にこのモデルを適用します。

<h2> 単純なランダム分割（再現性なし） </h2>

In [ ]:
compute_alpha = """
#standardSQL
SELECT
    SAFE_DIVIDE(
    SUM(arrival_delay * departure_delay),
    SUM(departure_delay * departure_delay)) AS alpha
FROM
(
    SELECT
        RAND() AS splitfield,
        arrival_delay,
        departure_delay
    FROM
        `bigquery-samples.airline_ontime_data.flights`
    WHERE
        departure_airport = 'DEN'
        AND arrival_airport = 'LAX'
)
WHERE
    splitfield < 0.8
"""

In [ ]:
results = bigquery.Client().query(compute_alpha).to_dataframe()
alpha = results["alpha"][0]
print(alpha)

<h3> 以下のようにトレーニング データとテスト データで RMSE を計算することの何が問題なのでしょうか？ </h3>

In [ ]:
compute_rmse = """
#standardSQL
SELECT
    dataset,
    SQRT(
        AVG(
            (arrival_delay - ALPHA * departure_delay) *
            (arrival_delay - ALPHA * departure_delay)
        )
    ) AS rmse,
    COUNT(arrival_delay) AS num_flights
FROM (
    SELECT
        IF (RAND() < 0.8, 'train', 'eval') AS dataset,
        arrival_delay,
        departure_delay
    FROM
        `bigquery-samples.airline_ontime_data.flights`
    WHERE
        departure_airport = 'DEN'
        AND arrival_airport = 'LAX' )
GROUP BY
    dataset
"""
bigquery.Client().query(
    compute_rmse.replace("ALPHA", str(alpha))
).to_dataframe()

ヒント:
* `compute_rmse` クエリでは、`compute_alpha` クエリと本当に同じトレーニング データを取得できていますか？
* `compute_alpha` と `compute_rmse` のブロックを再実行するたびに、同じ結果が得られますか？

<h3> 正しくトレーニングと評価を行うにはどうすればよいでしょうか？ </h3>
<br/>
実際のトレーニング データとホールドアウト（評価）データを使用して RMSE を計算する正しい方法がこちらです。これがどれほど難しく感じられるかに注目してください。

計算自体は正しくなりましたが、実験にはまだ再現性がありません。

何度か実行してみてください。毎回同じ結果が得られますか？

In [ ]:
train_and_eval_rand = """
#standardSQL
WITH
    alldata AS (
        SELECT
            IF (RAND() < 0.8, 'train', 'eval') AS dataset,
            arrival_delay,
            departure_delay
        FROM
            `bigquery-samples.airline_ontime_data.flights`
        WHERE
        departure_airport = 'DEN'
        AND arrival_airport = 'LAX' ),
    training AS (
        SELECT
            SAFE_DIVIDE(
                SUM(arrival_delay * departure_delay),
                SUM(departure_delay * departure_delay)) AS alpha
        FROM
            alldata
        WHERE
            dataset = 'train' )

SELECT
    MAX(alpha) AS alpha,
    dataset,
    SQRT(
        AVG(
            (arrival_delay - alpha * departure_delay) *
            (arrival_delay - alpha * departure_delay)
        )
    ) AS rmse,
    COUNT(arrival_delay) AS num_flights
FROM
    alldata,
    training
GROUP BY
    dataset
"""

In [ ]:
bigquery.Client().query(train_and_eval_rand).to_dataframe()

<h2> 日付のハッシュ（HASH）を使用したデータの分割 </h2>

日付ごとに分割してトレーニングしましょう。

In [ ]:
compute_alpha = """
#standardSQL
SELECT
    SAFE_DIVIDE(
        SUM(arrival_delay * departure_delay),
        SUM(departure_delay * departure_delay)) AS alpha
FROM
    `bigquery-samples.airline_ontime_data.flights`
WHERE
    departure_airport = 'DEN'
    AND arrival_airport = 'LAX'
    AND ABS(MOD(FARM_FINGERPRINT(date), 10)) < 8
"""
results = bigquery.Client().query(compute_alpha).to_dataframe()
alpha = results["alpha"][0]
print(alpha)

この $\alpha$ を使用して RMSE を計算できるようになりました。$\alpha$ の値は再現可能であるため、`compute_rmse` 内の $\alpha$ が `compute_alpha` で計算された $\alpha$ と異なってしまう心配はありません。

In [ ]:
compute_rmse = """
#standardSQL
SELECT
    IF(ABS(MOD(FARM_FINGERPRINT(date), 10)) < 8, 'train', 'eval') AS dataset,
    SQRT(
        AVG(
            (arrival_delay - ALPHA * departure_delay) *
            (arrival_delay - ALPHA * departure_delay)
        )
    ) AS rmse,
    COUNT(arrival_delay) AS num_flights
FROM
    `bigquery-samples.airline_ontime_data.flights`
WHERE
    departure_airport = 'DEN'
    AND arrival_airport = 'LAX'
GROUP BY
    dataset
"""
print(
    bigquery.Client()
    .query(compute_rmse.replace("ALPHA", str(alpha)))
    .to_dataframe()
    .head()
)

また、正しく分割を行った場合、評価データセットの RMSE がトレーニング データセットの RMSE からより大きく乖離することにも注目してください。これは当然予想されることです。`RAND()` の場合、同日のフライト間には強い相関があるため、トレーニング データセットと評価データセットの間でデータ漏洩（リーク）が発生していました。
<p>
これは、機械学習の分割を誤った方法で行うことの最大の危険性の 1 つです。つまり、<b>モデルが実際よりも優れているという誤った自信（過信）を抱いてしまうことになります！</b>

Copyright 2021 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.